In [29]:
"""
Импорт необходимых библиотек.
pandas - для работы с табличными данными.
numpy - для работы с массивами.
scipy.sparse - для создания разреженных матриц CSR.
implicit - библиотека для работы с коллаборативной фильтрацией (ALS и др.).
tqdm - библиотека для отображения индикаторов прогресса.
"""
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import implicit
from tqdm import tqdm

tqdm.pandas()

In [30]:
"""
Загрузка данных.
Загружаем файл с рейтингами (ratings_small.csv) и очищенные метаданные фильмов (movies_cleaned.csv).
Нас интересуют колонки userId, movieId и rating.
Необходимо привести movieId в ratings_small.csv и id в movies_cleaned.csv к одному типу (int) для корректного объединения.
"""
ratings = pd.read_csv('data/ratings_small.csv', usecols=['userId', 'movieId', 'rating'])
movies = pd.read_csv('data/movies_cleaned.csv')

ratings['movieId'] = ratings['movieId'].fillna(0).astype('int')
movies['id'] = movies['id'].fillna(0).astype('int')

In [31]:
"""
Фильтрация фильмов.
В рейтингах есть item-ы, которых нет в movies_cleaned.csv.
Оставляем только те фильмы, которые есть в базе метаданных.
"""
ratings = ratings[ratings['movieId'].isin(movies['id'])]

In [32]:
"""
Создание маппинга (отображения) идентификаторов.
Для создания разреженной матрицы нам нужны непрерывные индексы от 0 до N.
userId и movieId могут иметь пропуски в индексации.
user_map (userId -> index)
movie_map (movieId -> index)
user_inv_map (index -> userId)
movie_inv_map (index -> movieId)
"""
user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_map = {user_id: idx for idx, user_id in enumerate(user_ids)}
movie_map = {movie_id: idx for idx, movie_id in enumerate(movie_ids)}

user_inv_map = {idx: user_id for user_id, idx in user_map.items()}
movie_inv_map = {idx: movie_id for movie_id, idx in movie_map.items()}

ratings['user_idx'] = ratings['userId'].map(user_map)
ratings['movie_idx'] = ratings['movieId'].map(movie_map)

print(f"Количество пользователей: {len(user_map)}")
print(f"Количество фильмов: {len(movie_map)}")

Количество пользователей: 671
Количество фильмов: 2830


In [33]:
"""
Создание разреженной матрицы (CSR) взаимодействий.
Строки - фильмы (Items), столбцы - пользователи (Users).
Значения - рейтинг.
Для обучения implicit, как правило, подается item-user матрица.
"""
rows = ratings['movie_idx'].values
cols = ratings['user_idx'].values
values = ratings['rating'].values

item_user_matrix = csr_matrix((values, (rows, cols)), shape=(len(movie_map), len(user_map)))
item_user_matrix.eliminate_zeros()
print(f"Размерность матрицы: {item_user_matrix.shape}")
print(f"Плотность матрицы: {item_user_matrix.nnz / (item_user_matrix.shape[0] * item_user_matrix.shape[1]):.4f}")

Размерность матрицы: (2830, 671)
Плотность матрицы: 0.0237


In [34]:
"""
Инициализация модели AlternatingLeastSquares (ALS).

Параметры модели:
factors: Количество скрытых факторов (латентных размерностей), описывающих пользователей и объекты. Чем больше, тем точнее, но выше риск переобучения. По умолчанию 50.
regularization: Коэффициент регуляризации. Добавляется в функцию ошибки для предотвращения переобучения (слишком больших весов). По умолчанию 0.01.
iterations: Количество итераций алгоритма ALS. Больше итераций - потенциально более точное решение, но больше время обучения. По умолчанию 15.
random_state: Зерно генерации случайных чисел для воспроизводимости начальной инициализации факторов.
"""
model = implicit.als.AlternatingLeastSquares(factors=75, regularization=0.01, iterations=15, random_state=42)

In [35]:
"""
Обучение модели.
На вход метода fit подается матрица взаимодействий пользователей и объектов.
Мы используем созданную ранее item_user_matrix (Items x Users).
implicit уже автоматически отображает прогресс (tqdm), если он установлен.
"""
model.fit(item_user_matrix)

  0%|          | 0/15 [00:00<?, ?it/s]

In [36]:
"""
Функция рекомендаций.
Принимает идентификатор пользователя (из исходного датасета) и количество рекомендаций.
Возвращает список названий фильмов.
"""
def get_collaborative_recommendations(target_user_id, top_k=5):
    """
    Проверка наличия пользователя в данных.
    Если пользователя нет, возвращаем пустой список.
    """
    if target_user_id not in user_map:
        return []

    user_index = user_map[target_user_id]
    
    """
    Получение рекомендаций.
    Метод recommend принимает user_index и user_items матрицу для исключения уже просмотренных фильмов.
    user_items - это sparse matrix user x item (транспонированная item_user).
    """
    user_items = item_user_matrix.T.tocsr()
    
    """
    Используем метод recommend из библиотеки implicit.
    Он возвращает список идентификаторов рекомендованных объектов и, опционально, их оценки (scores).
    """
    recommendations = model.recommend(user_index, user_items[user_index], N=top_k)

    """
    Обработка результата model.recommend.
    В зависимости от версии библиотеки implicit формат возвращаемого значения может отличаться.
    В новых версиях возвращается кортеж (ids, scores).
    """
    if isinstance(recommendations, tuple):
        movie_indices = recommendations[0]
    elif len(recommendations) > 0 and isinstance(recommendations[0], (list, tuple, np.ndarray)):
        """
        Если вернулся список кортежей (например, для старых версий), берем первый элемент (индекс).
        """
        movie_indices = [r[0] for r in recommendations]
    else:
        """
        Если вернулся простой список индексов.
        """
        movie_indices = recommendations

    recommended_titles = []
    
    """
    Преобразование внутренних индексов фильмов обратно в названия фильмов.
    Используем movie_inv_map для получения movie_id и затем ищем название в DataFrame movies.
    """
    for idx in movie_indices:
        movie_id = movie_inv_map[idx]
        movie_row = movies[movies['id'] == movie_id]
        if not movie_row.empty:
            recommended_titles.append(movie_row['title'].values[0])
        else:
             recommended_titles.append(f"Movie ID {movie_id} (No title)")
            
    return recommended_titles

In [37]:
"""
Тестирование функции.
Выведем топ-5 фильмов с самыми высокими оценками для первых 10 пользователей.
Выведем top_k=10 рекомендаций от модели ALS в красивом текстовом формате.
"""
top_k = 10

"""
Получаем список первых 10 уникальных пользователей из нашего маппинга.
"""
first_10_users = list(user_map.keys())[:10]

for target_user in first_10_users:
    print(f"Пользователь с ID {target_user}:")

    """
    Проверка наличия пользователя в данных user_map.
    """
    if target_user in user_map:
        """
        Вывод топ-5 фильмов, оцененных пользователем.
        Сортируем список рейтингов по убыванию и выводим первые 5 фильмов.
        """
        user_movies = ratings[ratings['userId'] == target_user]
        top_user_movies = user_movies.sort_values(by='rating', ascending=False).head(5)
        
        print("\nТоп-5 фильмов, оцененных пользователем:")
        for _, row in top_user_movies.iterrows():
            movie_id = row['movieId']
            rating = row['rating']
            """
            Ищем название фильма в базе данных фильмов по его идентификатору.
            """
            movie_matches = movies[movies['id'] == movie_id]
            if not movie_matches.empty:
                title = movie_matches['title'].values[0]
                print(f"- {title} (Оценка: {rating})")
            else:
                print(f"- Movie ID {movie_id} (Без названия) (Оценка: {rating})")

        """
        Получение рекомендаций от модели ALS для заданного пользователя.
        """
        recs = get_collaborative_recommendations(target_user, top_k)
        
        """
        Вывод списка рекомендаций с использованием нумерации.
        """
        if recs:
            print(f"\nРекомендации от модели ALS:")
            for i, title in enumerate(recs, 1):
                print(f"{i}. {title}")
        else:
            print("\nНе удалось сформировать рекомендации.")
            
    else:
        """
        Если пользователь не найден, выводим соответствующее сообщение.
        """
        print("\nПользователь не найден в данных.")



Пользователь с ID 1:

Топ-5 фильмов, оцененных пользователем:
- American Pie (Оценка: 4.0)
- Rocky III (Оценка: 2.5)
- Confidentially Yours (Оценка: 2.5)
- My Tutor (Оценка: 2.0)
- Jay and Silent Bob Strike Back (Оценка: 2.0)

Рекомендации от модели ALS:
1. Rebecca
2. Mere Brother Ki Dulhan
3. Joshua
4. Reclaim Your Brain
5. The Hunchback of Notre Dame
6. The Wrong Trousers
7. Roustabout
8. One on Top of the Other
9. Read It and Weep
10. The Legend of Bagger Vance
Пользователь с ID 2:

Топ-5 фильмов, оцененных пользователем:
- The Dark (Оценка: 5.0)
- 48 Hrs. (Оценка: 5.0)
- Lili Marleen (Оценка: 5.0)
- Contempt (Оценка: 5.0)
- Berlin: Symphony of a Great City (Оценка: 5.0)

Рекомендации от модели ALS:
1. One on Top of the Other
2. Brief Encounter
3. The Legend of Bagger Vance
4. Highlander: The Final Dimension
5. Roustabout
6. Rope
7. Life Is a Long Quiet River
8. Kolya
9. The Elephant Man
10. 2 Days in Paris
Пользователь с ID 3:

Топ-5 фильмов, оцененных пользователем:
- The Million 